# Projet NOZ - Étape 4 : tester les hypothèses avec des statistiques

David DOE · 2026

But : les vues SQL de l'étape 3 ont montré des écarts (le samedi vend plus, le D.P.H. perd plus...).
Mais un écart peut venir du hasard. Ici, on mesure si chaque écart est réel ou s'il pourrait être dû au hasard.

Rappel important : les données sont simulées. Les résultats ne disent rien du vrai magasin NOZ.
Ils montrent que la méthode fonctionne : elle retrouve les effets mis dans la simulation.

| Hypothèse | Question | Test utilisé |
|---|---|---|
| H1 | Le CA dépend-il du jour de la semaine ? Est-il plus fort les jours de livraison ? | Kruskal-Wallis, Mann-Whitney |
| H2 | Le D.P.H. a-t-il plus de démarque inconnue que les autres rayons ? | Kruskal-Wallis, Mann-Whitney |
| H3 | La boutique (vitrine) a-t-elle moins de démarque inconnue ? | Mann-Whitney |
| H4 | Un magasin saturé à l'arrivée d'un lot renvoie-t-il plus de lots en RVN prématuré ? | Khi-deux (χ²) |

Installation (une seule fois) :
```
pip install -r requirements.txt
```

## 1. Se connecter à la base PostgreSQL

Python ne lit pas les CSV : il interroge directement la base `noz`, et même les vues de l'étape 3.
Les calculs sont définis une seule fois dans SQL, et Python les réutilise.

- `sqlalchemy` = la « prise » qui relie Python à PostgreSQL.
- `getpass` demande le mot de passe.

In [ ]:
import os, getpass
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sqlalchemy import create_engine, URL

# Si une adresse de connexion est deja definie (cas d'un test automatique), on l'utilise.
# Sinon, on demande le mot de passe de l'utilisateur "postgres".
url = os.environ.get("NOZ_DB_URL")
if url is None:
    url = URL.create("postgresql+psycopg2", username="postgres",
                     password=getpass.getpass("Mot de passe PostgreSQL : "),
                     host="localhost", port=5432, database="noz")
moteur = create_engine(url)

os.makedirs("figures", exist_ok=True)   # dossier ou on range les graphiques du rapport
SEUIL = 0.05                             # seuil de 5 % (explique juste en dessous)

# Petit test : si ca affiche 599, la connexion marche.
pd.read_sql("SELECT COUNT(*) AS nb_jours FROM magasin_jour", moteur)

## 2. Comprendre la « p-value » en 1 minute

On part toujours de l'idée « il n'y a aucune différence, c'est le hasard » (on l'appelle H0, l'hypothèse nulle).

La p-value répond à la question : *si c'était vraiment le hasard, quelle serait la probabilité de voir un écart aussi grand ?*

- p < 0,05 (moins de 5 %) : un tel écart serait très rare par hasard → on rejette H0 → l'écart est significatif (réel).
- p ≥ 0,05 : l'écart peut s'expliquer par le hasard → on ne peut pas conclure.

Exemple : une pièce tombe 10 fois de suite sur pile. Par hasard, ça arrive 1 fois sur 1 024 (p ≈ 0,001). On soupçonne alors une pièce truquée.

Pourquoi ces tests-là ? Les CA et les taux ne suivent pas une belle « courbe en cloche » (loi normale).
On utilise donc des tests non paramétriques (Kruskal-Wallis, Mann-Whitney) : ils comparent les rangs (le classement des valeurs) au lieu des moyennes, et ne demandent pas de courbe en cloche.

In [ ]:
def verdict(p):
    """Traduit une p-value en phrase simple."""
    if p < SEUIL:
        return f"p = {p:.2g} < 0,05 -> ecart SIGNIFICATIF (pas du hasard)"
    return f"p = {p:.2g} >= 0,05 -> ecart NON significatif (peut etre du hasard)"

## 3. H1 : le jour de la semaine et les livraisons

On lit la table `magasin_jour` (1 ligne = 1 jour, 599 jours) et la vue `v_ca_jour_semaine`.

Test 1 - Kruskal-Wallis : compare plusieurs groupes à la fois (ici les 7 jours). Question : *au moins un jour vend-il différemment des autres ?*

Test 2 - Mann-Whitney : compare deux groupes (jours avec livraison / jours sans). Question : *vend-on plus les jours de livraison ?*

In [ ]:
jours = pd.read_sql("SELECT date_jour, jour_semaine, ca_total_eur::float AS ca, nb_lots_recus "
                    "FROM magasin_jour", moteur)
vue_jours = pd.read_sql("SELECT * FROM v_ca_jour_semaine ORDER BY num_jour", moteur)
vue_jours

In [ ]:
# Test 1 : les 7 jours ont-ils le meme CA ?
groupes = [g["ca"] for _, g in jours.groupby("jour_semaine")]
h1_kw = stats.kruskal(*groupes)
print("Kruskal-Wallis (7 jours) :", verdict(h1_kw.pvalue))

# Test 2 : jours avec livraison contre jours sans livraison
avec = jours.loc[jours["nb_lots_recus"] > 0, "ca"]
sans = jours.loc[jours["nb_lots_recus"] == 0, "ca"]
h1_mw = stats.mannwhitneyu(avec, sans, alternative="two-sided")
print(f"CA moyen avec livraison : {avec.mean():.0f} EUR  |  sans livraison : {sans.mean():.0f} EUR")
print("Mann-Whitney (livraison oui/non) :", verdict(h1_mw.pvalue))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
couleurs = ["#c0392b" if j == "Samedi" else "#7f8c8d" for j in vue_jours["jour_semaine"]]
ax.bar(vue_jours["jour_semaine"], vue_jours["ca_moyen_eur"].astype(float), color=couleurs)
ax.axhline(jours["ca"].mean(), linestyle="--", color="black", linewidth=1, label="moyenne")
ax.set_ylabel("CA moyen par jour (EUR)")
ax.set_title("H1 - CA moyen par jour de la semaine (donnees simulees)")
ax.legend()
plt.tight_layout()
plt.savefig("figures/h1_ca_jour_semaine.png", dpi=150)
plt.show()

Comment lire H1 :
- Le jour de la semaine compte énormément (p minuscule) : le samedi est le plus fort, le dimanche le plus faible, comme dans l'annexe du mémoire.
- Les jours de livraison ne vendent pas plus (p bien au-dessus de 0,05). L'ancienne idée « on vend plus les jours de livraison » est rejetée : ce sont les clients qui font le pic (le samedi), pas les camions.

## 4. H2 et H3 : où disparaît la marchandise ?

On utilise les inventaires (1 ligne = 1 rayon à 1 date, 21 inventaires × 9 rayons).
C'est le seul endroit où, dans la vraie vie, on voit la démarque inconnue. On ne triche pas avec la vérité terrain.

- Kruskal-Wallis : les 9 rayons ont-ils le même taux de démarque ?
- Mann-Whitney : un rayon contre tous les autres (D.P.H. pour H2, Boutique pour H3), et D.P.H. contre Bien-être.

In [ ]:
inv = pd.read_sql("SELECT date_inventaire, categorie, taux_demarque_inconnue_pct::float AS taux "
                  "FROM inventaires", moteur)
pd.read_sql("SELECT * FROM v_demarque_categorie ORDER BY taux_demarque_pct DESC", moteur)

In [ ]:
h2_kw = stats.kruskal(*[g["taux"] for _, g in inv.groupby("categorie")])
print("Kruskal-Wallis (9 rayons) :", verdict(h2_kw.pvalue))

def un_contre_les_autres(rayon):
    a = inv.loc[inv["categorie"] == rayon, "taux"]
    b = inv.loc[inv["categorie"] != rayon, "taux"]
    test = stats.mannwhitneyu(a, b, alternative="two-sided")
    print(f"{rayon:<25} mediane {a.median():.2f} %  contre {b.median():.2f} % pour les autres  -> {verdict(test.pvalue)}")
    return test

h2_mw  = un_contre_les_autres("4. D.P.H.")
h3_mw  = un_contre_les_autres("BOUTIQUE MANON MERCIER")
_      = un_contre_les_autres("6. Bien-être")

dph = inv.loc[inv["categorie"] == "4. D.P.H.", "taux"]
bien = inv.loc[inv["categorie"] == "6. Bien-être", "taux"]
h2_duel = stats.mannwhitneyu(dph, bien, alternative="two-sided")
print("\nD.P.H. contre Bien-etre :", verdict(h2_duel.pvalue))

In [ ]:
ordre = inv.groupby("categorie")["taux"].median().sort_values(ascending=False).index
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.boxplot([inv.loc[inv["categorie"] == c, "taux"] for c in ordre])
ax.set_xticks(range(1, len(ordre) + 1), ordre)
ax.set_ylabel("Taux de demarque inconnue (% du CA)")
ax.set_title("H2 / H3 - Demarque inconnue par rayon, 21 inventaires (donnees simulees)")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig("figures/h2_h3_demarque_rayon.png", dpi=150)
plt.show()

Lire une boîte à moustaches : le trait au milieu = la médiane (la valeur du milieu) ; la boîte = la moitié centrale des inventaires ; les moustaches = l'étendue ; les points = valeurs isolées.

Comment lire H2 et H3 :
- H2 partiellement confirmée : le D.P.H. perd nettement plus que la moyenne… mais pas plus que le Bien-être (p ≈ 0,97 entre les deux : impossible de les départager). Il faut donc reformuler : *« le D.P.H. et le Bien-être sont les rayons les plus touchés »*.
- H3 confirmée : la boutique a la démarque la plus faible, et l'écart est significatif. Mais ce n'est pas zéro : on écrit « nettement plus faible », pas « quasi nulle ».

## 5. H4 : la saturation du magasin et les RVN prématurés

On utilise la vue `v_vie_lot` (1 ligne = 1 lot), seulement les lots analysables (vie complète).
On compare le pourcentage de RVN prématuré selon la saturation du magasin le jour où le lot arrive (vue `v_saturation_rvn`).

Test du Khi-deux (χ²) : il sert pour des comptages (oui/non). Il compare ce qu'on observe à ce qu'on verrait si la saturation n'avait aucun lien avec le RVN.

In [ ]:
sat = pd.read_sql("SELECT * FROM v_saturation_rvn ORDER BY tranche_saturation", moteur)
sat

In [ ]:
# Tableau de comptage : pour chaque tranche, [nb RVN premature, nb autres lots]
tableau = [[r.nb_rvn_premature, r.nb_lots - r.nb_rvn_premature] for r in sat.itertuples()]
h4_chi2 = stats.chi2_contingency(tableau)
print(f"Khi-deux = {h4_chi2.statistic:.1f}  ->", verdict(h4_chi2.pvalue))

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(sat["tranche_saturation"], sat["pct_rvn_premature"].astype(float), color="#2c3e50")
ax.set_ylabel("% de lots en RVN premature")
ax.set_xlabel("Saturation du magasin a l'arrivee du lot")
ax.set_title("H4 - Saturation et RVN premature (donnees simulees)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("figures/h4_saturation_rvn.png", dpi=150)
plt.show()

Comment lire H4 :
- Le lien est significatif : plus le magasin est plein à l'arrivée, plus le lot finit souvent en RVN prématuré (de 7-8 % à 15 %).
- Mais il est modéré : même au-dessus de 1,10, 85 % des lots ne sont pas renvoyés. La saturation seule ne suffit pas à prédire le RVN : c'est pour ça qu'on construira un modèle avec plusieurs variables à l'étape suivante.
- Rappel de la v1 : la règle « saturation > 1,10 → RVN » était presque parfaite, ce qui était irréaliste.

## 6. Bonus : la méthode retrouve-t-elle la vérité cachée ?

C'est l'avantage d'un simulateur : on connaît la vérité terrain (les vrais vols). On peut donc vérifier si les inventaires (ce qu'un magasin voit vraiment) racontent la même histoire.

⚠️ La vérité terrain sert uniquement à vérifier, jamais à analyser ou à prédire.

In [ ]:
verite = pd.read_sql("""
    SELECT a.categorie,
           ROUND(10000.0 * SUM(t.vol_reel)           / SUM(v.stock_theorique_debut_jour), 1) AS vols_pour_10000,
           ROUND(10000.0 * SUM(t.casse_non_declaree) / SUM(v.stock_theorique_debut_jour), 1) AS casse_cachee_pour_10000
    FROM verite_terrain_lot_jour t
    JOIN ventes_lot_jour v USING (date_jour, arrivage_id)
    JOIN arrivages a       USING (arrivage_id)
    GROUP BY a.categorie
    ORDER BY vols_pour_10000 DESC""", moteur)
verite

Lecture : on compte les vols pour 10 000 articles en rayon par jour.
- Les vrais vols sont les plus forts en D.P.H. et Bien-être et les plus faibles à la boutique : les inventaires ont vu juste pour H2 et H3.
- Les Boissons ont peu de vols, mais beaucoup de casse cachée (bouteilles cassées non déclarées). C'est pour ça qu'elles sont 3e en démarque inconnue.
- Leçon pour le rapport : la démarque inconnue d'un inventaire mélange vols et casse non déclarée. Un taux élevé ne veut pas forcément dire « vol ».

## 7. Synthèse (à recopier dans le rapport, section 5.1)

In [ ]:
synthese = pd.DataFrame([
    ["H1", "Le CA varie selon le jour (pic le samedi)",       "Kruskal-Wallis", h1_kw.pvalue,   "Confirmee"],
    ["H1 bis", "On vend plus les jours de livraison",       "Mann-Whitney",   h1_mw.pvalue,   "Rejetee"],
    ["H2", "D.P.H. : plus de demarque que les autres rayons", "Mann-Whitney",   h2_mw.pvalue,   "Partielle (a egalite avec Bien-etre)"],
    ["H3", "Boutique : moins de demarque",                  "Mann-Whitney",   h3_mw.pvalue,   "Confirmee (faible, pas nulle)"],
    ["H4", "Saturation a l'arrivee -> plus de RVN premature", "Khi-deux",       h4_chi2.pvalue, "Confirmee (lien modere)"],
], columns=["hypothese", "enonce", "test", "p_value", "verdict"])
synthese["p_value"] = synthese["p_value"].map(lambda p: f"{p:.1e}")
synthese

Phrase à retenir pour le rapport et l'entretien :
> « Sur des données simulées, le pipeline SQL + Python retrouve les effets introduits dans la simulation. Les tests non paramétriques confirment le pic du samedi, rejettent l'effet "jour de livraison", et montrent que la démarque inconnue d'un inventaire mélange vols et casse non déclarée. »

Prochaine étape : prédire le RVN prématuré avec un modèle (régression logistique, puis Random Forest), avec un découpage temporel pour éviter la fuite de données.